# Financial Investment Data Quality & Analytics

## Objective
Validate customer, transaction and branch data for completeness, uniqueness, validity, consistency and referential integrity before financial analysis.

In [ ]:
import pandas as pd
import numpy as np
branch=pd.read_csv("../data/raw/branch.csv")
customer=pd.read_csv("../data/raw/customer.csv")
transaction=pd.read_csv("../data/raw/transaction.csv")
transaction["Transaction_Date"]=pd.to_datetime(transaction["Transaction_Date"],dayfirst=True)
print("Branch:",branch.shape)
print("Customer:",customer.shape)
print("Transaction:",transaction.shape)


## Data Quality

In [ ]:
for name,df in [("Branch",branch),("Customer",customer),("Transaction",transaction)]:
    print("\n"+name)
    display(df.isna().sum().to_frame("Missing"))
    print("Duplicate rows:",df.duplicated().sum())


## Referential Integrity

In [ ]:
print("Customer -> Branch orphan records:",(~customer.Branch_ID.isin(branch.Branch_ID)).sum())
print("Transaction -> Customer orphan records:",(~transaction.Customer_ID.isin(customer.Customer_ID)).sum())


## Business Rule Checks

In [ ]:
checks={
    "Negative branch expenses":int((branch.Expenses<0).sum()),
    "Invalid customer ages":int((customer.Age.notna() & ((customer.Age<18)|(customer.Age>100))).sum()),
    "Negative transaction amounts":int((transaction.Transaction_Amount<0).sum()),
    "Negative investment amounts":int((transaction.Investment_Amount<0).sum())
}
pd.Series(checks,name="Exception_Count")


## Master Dataset

In [ ]:
master=(transaction
 .merge(customer,on="Customer_ID",how="left",validate="many_to_one")
 .merge(branch[["Branch_ID","Firm_Revenue","Expenses","Profit_Margin"]],on="Branch_ID",how="left",validate="many_to_one"))
master["Year"]=master.Transaction_Date.dt.year
master["Month"]=master.Transaction_Date.dt.to_period("M").astype(str)
master.head()


## Investment Analysis

In [ ]:
master.groupby("Investment_Type").agg(
    Transactions=("Transaction_ID","count"),
    Investment_Value=("Investment_Amount","sum"),
    Avg_Investment=("Investment_Amount","mean")
).sort_values("Investment_Value",ascending=False)


## Regional Analysis

In [ ]:
master.groupby("Region").agg(
    Customers=("Customer_ID","nunique"),
    Transactions=("Transaction_ID","count"),
    Transaction_Value=("Transaction_Amount","sum"),
    Investment_Value=("Investment_Amount","sum"),
    Avg_Balance=("Total_Balance","mean")
).sort_values("Transaction_Value",ascending=False)


## Branch Performance

In [ ]:
master.groupby(["Branch_ID","City","Region"]).agg(
    Customers=("Customer_ID","nunique"),
    Transactions=("Transaction_ID","count"),
    Transaction_Value=("Transaction_Amount","sum"),
    Investment_Value=("Investment_Amount","sum"),
    Firm_Revenue=("Firm_Revenue","first"),
    Expenses=("Expenses","first"),
    Profit_Margin=("Profit_Margin","first")
).reset_index().sort_values("Transaction_Value",ascending=False).head(10)
